# 05 — The Qwen3 decoder block (M1.5)

One transformer block = two **pre-norm residual sublayers**: causal self-attention,
then the SwiGLU MLP. This notebook uses `qwen3_scratch.layers.Qwen3DecoderLayer`,
which only wires the verified components from notebooks 01–04. CPU only, no
downloads. Plots need `matplotlib`.

Write-up: `docs/components/decoder.md`.

## 1. From the original Transformer to Qwen3

| architecture | per block | normalisation |
|---|---|---|
| Transformer decoder (2017, encoder-decoder) | masked self-attn → **cross-attn to encoder** → FFN | **post-norm**: `LN(x + f(x))` |
| decoder-only LM (GPT) | masked self-attn → FFN | GPT-2+: **pre-norm** `x + f(LN(x))` |
| **Qwen3** | causal GQA self-attn (QK-Norm, RoPE) → SwiGLU | pre-norm with **RMSNorm** |

A decoder-only LM has no encoder, so there is nothing to cross-attend to: the
"source" and "target" are the same token stream.

$$h_1 = x + \mathrm{Attn}(\mathrm{RMSNorm}_1(x)),\qquad
y = h_1 + \mathrm{SwiGLU}(\mathrm{RMSNorm}_2(h_1))$$

- **Attention** moves information *between* positions (which earlier tokens matter).
- **SwiGLU** transforms each position *independently* (what to compute from it).
- **Residuals** make each sublayer an *update* added to the stream.
- **Pre-norm**: each sublayer reads a normalised copy; the stream itself keeps its scale.

In [ ]:
import torch

from qwen3_scratch.config import QWEN3_0_6B_BASE, TINY_QWEN3
from qwen3_scratch.layers import (
    Qwen3DecoderLayer,
    RotaryEmbedding,
    bool_to_additive_mask,
    causal_mask,
    make_position_ids,
)

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

_ = torch.manual_seed(0)
cfg = TINY_QWEN3
layer = Qwen3DecoderLayer.from_config(cfg)
rope = RotaryEmbedding.from_config(cfg)
print(layer)

## 2. Tensor shapes through the block

In [ ]:
B, T = 2, 5
x = torch.randn(B, T, cfg.hidden_size)
cos, sin = rope(make_position_ids(T))
y, tr = layer(x, cos, sin, return_trace=True)
rows = [
    ("x (residual stream)", x),
    ("input_layernorm(x)", tr.input_norm),
    ("  q heads after RoPE", tr.attention.q_rope),
    ("  k heads after RoPE", tr.attention.k_rope),
    ("  attention probs", tr.attention.probs),
    ("self_attn output", tr.attn_output),
    ("h1 = x + attn", tr.residual_1),
    ("post_attention_layernorm(h1)", tr.post_attn_norm),
    ("mlp output", tr.mlp_output),
    ("y = h1 + mlp", tr.output),
]
for name, t in rows:
    print(f"{name:30s} {tuple(t.shape)}")

## 3. Residual connections: identity paths

$\partial y/\partial x = I + \partial\,\mathrm{Attn}/\partial x + \ldots$: the identity
term carries the gradient straight through, however deep the stack. If both
sublayers output zero, the block is exactly the identity.

In [ ]:
ablated = Qwen3DecoderLayer.from_config(cfg)
ablated.load_state_dict(layer.state_dict())
with torch.no_grad():
    ablated.self_attn.o_proj.weight.zero_()
    ablated.mlp.down_proj.weight.zero_()
print("block == identity when both sublayers output 0:", torch.equal(ablated(x, cos, sin), x))

xg = x.clone().requires_grad_()
u = torch.randn_like(x)
ablated(xg, cos, sin).backward(u)
print("gradient == upstream (identity Jacobian):", torch.equal(xg.grad, u))

## 4. Pre-norm vs post-norm in a deep stack

We stack up to 24 blocks with random weights and compare the pre-norm wiring
(Qwen3) with a post-norm wiring built from the *same* submodules,
`h = RMSNorm(h + f(h))`. For each depth $k$ we measure the norm of the gradient
that reaches the input from a fixed linear loss $\langle y_k, r\rangle$, and the
size of the residual stream. Sublayer weights use std 0.2, which mimics blocks
whose updates are not tiny. With std 0.02 (a fresh model) both wirings behave
almost identically, because every update is small.

In [ ]:
def make_stack(n, std):
    _ = torch.manual_seed(1)
    blocks = [Qwen3DecoderLayer.from_config(cfg) for _ in range(n)]
    for blk in blocks:
        for name, p in blk.named_parameters():
            if "norm" not in name:
                torch.nn.init.normal_(p, std=std)
    return blocks


def pre_norm(blocks, h):
    for blk in blocks:
        h = blk(h, cos, sin)
    return h


def post_norm(blocks, h):
    for blk in blocks:
        h = blk.input_layernorm(h + blk.self_attn(h, cos, sin))
        h = blk.post_attention_layernorm(h + blk.mlp(h))
    return h


blocks = make_stack(24, std=0.2)
r = torch.randn(B, T, cfg.hidden_size, generator=torch.Generator().manual_seed(3))
depths = list(range(1, 25))
results = {}
for name, fn in [("pre-norm (Qwen3)", pre_norm), ("post-norm", post_norm)]:
    grads, norms = [], []
    for k in depths:
        h = x.clone().requires_grad_()
        out = fn(blocks[:k], h)
        (out * r).sum().backward()
        grads.append(h.grad.norm().item())
        norms.append(out.norm(dim=-1).mean().item())
    results[name] = (grads, norms)
    print(
        f"{name:18s} input-grad norm at depth 1/12/24: "
        f"{grads[0]:.3g} / {grads[11]:.3g} / {grads[23]:.3g};  "
        f"stream norm: {norms[0]:.3g} / {norms[11]:.3g} / {norms[23]:.3g}"
    )

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    for (name, (grads, norms)), color in zip(results.items(), ["#2a78d6", "#eb6834"], strict=True):
        axes[0].semilogy(depths, grads, lw=2, color=color, label=name)
        axes[1].plot(depths, norms, lw=2, color=color, label=name)
    axes[0].set_title("gradient norm at the input", fontsize=10)
    axes[1].set_title("mean norm of the output stream", fontsize=10)
    for ax in axes:
        ax.set_xlabel("number of stacked blocks")
        ax.grid(alpha=0.25, lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].legend(frameon=False)
    plt.tight_layout()
    plt.show()

What the numbers show (for this random stack, not a general proof):

- **Post-norm** renormalises the stream after every sublayer, so the stream norm is
  pinned at $\sqrt H$. The identity path is interrupted by a normalisation at each
  step, and the gradient at the input changes much more steeply with depth.
- **Pre-norm** keeps a clean identity path from input to output, so the gradient
  grows smoothly with depth. The stream itself grows, because every block *adds*
  to it. That is why Qwen3 applies a **final RMSNorm** before the LM head (M1.6).

Xiong et al. (2020, *On Layer Normalization in the Transformer Architecture*)
analyse this effect and show that pre-norm models train stably without long
learning-rate warm-up.

## 5. Forward and backward through the block

In [ ]:
layer.zero_grad()
xb = torch.randn(1, 6, cfg.hidden_size, requires_grad=True)
out = layer(xb, *rope(make_position_ids(6)))
out[0, 3].sum().backward()
print(
    "grad norm per input position (loss at position 3):",
    [round(v, 4) for v in xb.grad[0].norm(dim=-1).tolist()],
)
for name, p in layer.named_parameters():
    print(f"  {name:32s} {tuple(p.shape)!s:14s} grad norm {p.grad.norm():.4f}")

## 6. Hugging Face comparison

In [ ]:
try:
    import transformers
    from transformers.models.qwen3 import modeling_qwen3 as hf_mod
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf_cfg = transformers.Qwen3Config(
        hidden_size=cfg.hidden_size,
        intermediate_size=cfg.intermediate_size,
        num_attention_heads=cfg.num_attention_heads,
        num_key_value_heads=cfg.num_key_value_heads,
        head_dim=cfg.head_dim,
        rope_theta=cfg.rope_theta,
        rms_norm_eps=cfg.rms_norm_eps,
        attn_implementation="eager",
    )
    hf = hf_mod.Qwen3DecoderLayer(hf_cfg, layer_idx=0).eval()
    hf.load_state_dict(layer.state_dict())  # same parameter names
    pos = make_position_ids(T, batch_size=B)
    hcos, hsin = hf_mod.Qwen3RotaryEmbedding(hf_cfg)(x, pos)
    mask = bool_to_additive_mask(causal_mask(pos, pos), torch.float32)[:, None]
    with torch.no_grad():
        h_out = hf(x, attention_mask=mask, position_embeddings=(hcos, hsin), position_ids=pos)
        h_out = h_out[0] if isinstance(h_out, tuple) else h_out
    print("decoder output bit-identical to HF:", torch.equal(h_out, layer(x, cos, sin)))

## 7. Common mistakes

- Post-norm wiring (`norm(x + f(x))`) or normalising the residual itself.
- Feeding the *normalised* tensor into the residual add (`h1 = norm(x) + attn`).
- Swapping `input_layernorm` and `post_attention_layernorm` (both have shape `(H,)`,
  so loading still succeeds silently; parity tests catch it).
- Running the MLP before attention, or adding an encoder cross-attention.
- Adding absolute position embeddings to `x`: Qwen3 positions live only in RoPE
  inside attention.
- Recomputing RoPE tables with different positions in different layers: all layers
  share one `(cos, sin)` per forward.

### Review questions
1. Write the Jacobian of one pre-norm block and point out the identity term.
2. Why does a pre-norm stack need a final normalisation before the LM head?
3. Which sublayer mixes information across positions, and which does not?
4. How many parameters does one Qwen3-0.6B block have, and how are they split?

In [ ]:
big = Qwen3DecoderLayer.from_config(QWEN3_0_6B_BASE, device="meta")
parts = {}
for name, p in big.named_parameters():
    parts[name.split(".")[0]] = parts.get(name.split(".")[0], 0) + p.numel()
print({k: f"{v:,}" for k, v in parts.items()}, "total", f"{sum(parts.values()):,}")